# Una solución empresarial completa

## Ahora llevaremos nuestro proyecto del día 1 al siguiente nivel

### DESAFÍO EMPRESARIAL:

Crear un producto que genere un folleto para una empresa que se utilizará para posibles clientes, inversores y posibles reclutas.

Se nos proporcionará un nombre de empresa y su sitio web principal.

Consulte el final de este cuaderno para ver ejemplos de aplicaciones empresariales del mundo real.

Y recuerde: ¡siempre estoy disponible si tiene problemas o ideas! No dude en comunicarse conmigo.

In [ ]:
# imports
# Si esto falla, verifica que esté ejecutándose desde un entorno "activado" con (llms) en el símbolo del sistema

import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [ ]:
# Inicialización y constantes and constants

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key[:8]=='sk-proj-':
    print("La clave de API parece buena")
else:
    print("¿Puede haber un problema con tu clave API? ¡Visita el cuaderno de resolución de problemas!")
    
MODEL = 'gpt-4o-mini'
openai = OpenAI()

La clave de API parece buena


In [ ]:
# La clase para representar una Página Web

class Website:
    """
    Una clase de utilidad para representar un sitio web que hemos scrappeado, ahora con enlaces
    """

    def __init__(self, url):
        self.url = url
        response = requests.get(url)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "Sin título"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Título de la Web:\n{self.title}\nContenido de la Web:\n{self.text}\n\n"

In [ ]:
frog = Website("https://cursos.frogamesformacion.com")
print(frog.get_contents())
frog.links

Título de la Web:
Frogames
Contenido de la Web:
Ir al contenido principal
Frogames
Menú alternativo
Menú
Iniciar sesión
Ganadora del premio 'Enseñanza online de datos y competencias digitales más innovadora de Europa, 2023'
Pasión por
aprender
con los
mejores
En Frogames Formación te ayudamos a convertirte en todo un experto en: Programación de Videojuegos, Inteligencia Artificial, Machine Learning, Desarrollo de Apps, Data Science y mucho más.
Aprende mientras te diviertes
Cursos, Rutas y Suscripciones
Certificados de finalización
Qué encontrarás
dentro
de Frogames
Cursos online y formación de calidad para toda la família
Rutas temáticas
Rutas organizadas para que aprendas paso a paso, subiendo cada escalón e incrementando tus conocimientos adquiridos
Instructores Expertos
Con un equipo de profesionales y expertos en la materia que te acompañará a lo largo de todo el aprendizaje en la plataforma
Certificados blockchain
Títulos verificados por blockchain para cada habilidad que aprenda

['#main-content',
 '/',
 '/',
 '/users/sign_in',
 'https://cursos.frogamesformacion.com/pages/rutas',
 'https://cursos.frogamesformacion.com/pages/instructores',
 'https://cursos.frogamesformacion.com/pages/certificaciones',
 'https://cursos.frogamesformacion.com/collections',
 '/courses/geometria-analitica-desde-cero',
 '/courses/geometria-analitica-desde-cero',
 '/courses/agentes-ia',
 '/courses/agentes-ia',
 '/courses/domina-android-desde-cero-kotlin-compose',
 '/courses/domina-android-desde-cero-kotlin-compose',
 '/courses/unity-ui',
 '/courses/unity-ui',
 '/courses/matematicas-ml-3',
 '/courses/matematicas-ml-3',
 '/courses/flutter-firebase-ai',
 '/courses/flutter-firebase-ai',
 '/courses/google-classroom',
 '/courses/google-classroom',
 '/courses/ia-productividad',
 '/courses/ia-productividad',
 '/courses/bandas-sonoras-4',
 '/courses/bandas-sonoras-4',
 '/courses/musica-videojuegos',
 '/courses/musica-videojuegos',
 '/courses/bandas-sonoras-3',
 '/courses/bandas-sonoras-3',
 '/c

## Primer paso: hacer que GPT-4o-mini determine qué enlaces son relevantes

### Usar una llamada a gpt-4o-mini para leer los enlaces en una página web y responder en JSON estructurado.
Debería decidir qué enlaces son relevantes y reemplazar los enlaces relativos como "/about" con "https://company.com/about".
Usaremos "one shot prompting" en las que proporcionamos un ejemplo de cómo debería responder en la solicitud.

Este es un excelente caso de uso para un LLM, porque requiere una comprensión matizada. Imagínate intentar programar esto sin LLMs analizando la página web: ¡sería muy difícil!

Nota al margen: existe una técnica más avanzada llamada "Salidas estructuradas" en la que requerimos que el modelo responda de acuerdo con una especificación. Cubrimos esta técnica en la Semana 8 durante nuestro proyecto autónomo de inteligencia artificial Agentic.

In [ ]:
link_system_prompt = "Se te proporciona una lista de enlaces que se encuentran en una página web. \
Puedes decidir cuáles de los enlaces serían los más relevantes para incluir en un folleto sobre la empresa, \
como enlaces a una página Acerca de, una página de la empresa, las carreras/empleos disponibles o páginas de Cursos/Packs.\n"
link_system_prompt += "Debes responder en JSON como en este ejemplo:"
link_system_prompt += """
{
    "links": [
        {"type": "Pagina Sobre nosotros", "url": "https://url.completa/aqui/va/sobre/nosotros"},
        {"type": "Pagina de Cursos": "url": "https://otra.url.completa/courses"}
    ]
}
"""

In [ ]:
print(link_system_prompt)

Se te proporciona una lista de enlaces que se encuentran en una página web. Puedes decidir cuáles de los enlaces serían los más relevantes para incluir en un folleto sobre la empresa, como enlaces a una página Acerca de, una página de la empresa, las carreras/empleos disponibles o páginas de Cursos/Packs.
Debes responder en JSON como en este ejemplo:
{
    "links": [
        {"type": "Pagina Sobre nosotros", "url": "https://url.completa/aqui/va/sobre/nosotros"},
        {"type": "Pagina de Cursos": "url": "https://otra.url.completa/courses"}
    ]
}



In [ ]:
def get_links_user_prompt(website):
    user_prompt = f"Aquí hay una lista de enlaces de la página web {website.url} - "
    user_prompt += "Por favor, decide cuáles de estos son enlaces web relevantes para un folleto sobre la empresa. Responde con la URL https completa en formato JSON. \
No incluyas Términos y Condiciones, Privacidad ni enlaces de correo electrónico.\n"
    user_prompt += "Links (puede que algunos sean links relativos):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [ ]:
print(get_links_user_prompt(frog))

Aquí hay una lista de enlaces de la página web https://cursos.frogamesformacion.com - Por favor, decide cuáles de estos son enlaces web relevantes para un folleto sobre la empresa. Responde con la URL https completa en formato JSON. No incluyas Términos y Condiciones, Privacidad ni enlaces de correo electrónico.
Links (puede que algunos sean links relativos):
#main-content
/
/
/users/sign_in
https://cursos.frogamesformacion.com/pages/rutas
https://cursos.frogamesformacion.com/pages/instructores
https://cursos.frogamesformacion.com/pages/certificaciones
https://cursos.frogamesformacion.com/collections
/courses/geometria-analitica-desde-cero
/courses/geometria-analitica-desde-cero
/courses/agentes-ia
/courses/agentes-ia
/courses/domina-android-desde-cero-kotlin-compose
/courses/domina-android-desde-cero-kotlin-compose
/courses/unity-ui
/courses/unity-ui
/courses/matematicas-ml-3
/courses/matematicas-ml-3
/courses/flutter-firebase-ai
/courses/flutter-firebase-ai
/courses/google-classroom


In [ ]:
def get_links(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
      ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    return json.loads(result)

In [ ]:
anthropic = Website("https://anthropic.com")
anthropic.links

['#main',
 '#footer',
 'https://www.anthropic.com/',
 'https://www.anthropic.com/research',
 'https://www.anthropic.com/economic-futures',
 'https://www.anthropic.com/transparency',
 'https://www.anthropic.com/news/announcing-our-updated-responsible-scaling-policy',
 'http://trust.anthropic.com/',
 'https://www.anthropic.com/learn',
 'https://www.anthropic.com/engineering',
 'https://docs.claude.com',
 'https://www.anthropic.com/company',
 'https://www.anthropic.com/careers',
 'https://www.anthropic.com/events',
 'https://www.anthropic.com/news',
 'https://claude.ai',
 'https://claude.com/product/overview',
 'https://claude.com/product/claude-code',
 'https://claude.com/platform/api',
 'https://claude.com/pricing',
 'https://claude.com/contact-sales',
 'https://www.anthropic.com/claude/opus',
 'https://www.anthropic.com/claude/sonnet',
 'https://www.anthropic.com/claude/haiku',
 'https://claude.ai/login',
 'https://console.anthropic.com',
 '#',
 'https://claude.ai/login',
 'https://cla

In [ ]:
get_links("https://anthropic.com")

{'links': [{'type': 'Pagina Sobre nosotros',
   'url': 'https://www.anthropic.com/company'},
  {'type': 'Pagina de Carreras', 'url': 'https://www.anthropic.com/careers'},
  {'type': 'Pagina de Cursos', 'url': 'https://www.anthropic.com/learn'},
  {'type': 'Pagina de Investigación',
   'url': 'https://www.anthropic.com/research'},
  {'type': 'Pagina de Eventos', 'url': 'https://www.anthropic.com/events'}]}

In [ ]:
get_links("https://cursos.frogamesformacion.com")

{'links': [{'type': 'Pagina de Inicio',
   'url': 'https://cursos.frogamesformacion.com'},
  {'type': 'Pagina Rutas',
   'url': 'https://cursos.frogamesformacion.com/pages/rutas'},
  {'type': 'Pagina de Instructores',
   'url': 'https://cursos.frogamesformacion.com/pages/instructores'},
  {'type': 'Pagina de Certificaciones',
   'url': 'https://cursos.frogamesformacion.com/pages/certificaciones'},
  {'type': 'Pagina de Nuestros Clientes',
   'url': 'https://cursos.frogamesformacion.com/pages/nuestros-clientes'},
  {'type': 'Pagina para Empresas',
   'url': 'https://cursos.frogamesformacion.com/pages/frogames-para-empresas'},
  {'type': 'Pagina de Premios',
   'url': 'https://cursos.frogamesformacion.com/pages/premios'},
  {'type': 'Pagina de Afiliados',
   'url': 'https://cursos.frogamesformacion.com/pages/afiliados'},
  {'type': 'Pagina de Cursos',
   'url': 'https://cursos.frogamesformacion.com/collections'},
  {'type': 'Pagina de Facebook', 'url': 'https://www.facebook.com/froggames

## Segundo paso: ¡crea el folleto!

Reúne todos los detalles en otro mensaje para GPT4-o

In [ ]:
def get_all_details(url):
    result = "Landing page:\n"
    result += Website(url).get_contents()
    links = get_links(url)
    print("Links encontrados:", links)
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_contents()
    return result

In [ ]:
print(get_all_details("https://anthropic.com"))

Links encontrados: {'links': [{'type': 'Pagina Sobre nosotros', 'url': 'https://www.anthropic.com/company'}, {'type': 'Pagina de Carreras', 'url': 'https://www.anthropic.com/careers'}, {'type': 'Pagina de Cursos', 'url': 'https://www.anthropic.com/learn'}, {'type': 'Pagina de Investigación', 'url': 'https://www.anthropic.com/research'}, {'type': 'Pagina de Transparencia', 'url': 'https://www.anthropic.com/transparency'}, {'type': 'Pagina de Eventos', 'url': 'https://www.anthropic.com/events'}]}
Landing page:
Título de la Web:
Home \ Anthropic
Contenido de la Web:
Skip to main content
Skip to footer
Research
Economic Futures
Commitments
Initiatives
Transparency
Responsible Scaling Policy
Trust center
Security and compliance
Learn
Learn
Anthropic Academy
Engineering at Anthropic
Developer docs
Company
About
Careers
Events
News
Try Claude
Try Claude
Try Claude
Learn more about Claude
Products
Claude
Claude Code
Claude Developer Platform
Pricing
Contact sales
Models
Opus
Sonnet
Haiku
Log i

In [ ]:
system_prompt = "Eres un asistente que analiza el contenido de varias páginas relevantes del sitio web de una empresa\
y crea un folleto breve sobre la empresa para posibles clientes, inversores y nuevos empleados. Responde en formato Markdown.\
Incluye detalles sobre la cultura de la empresa, los clientes, las carreras/empleos y los cursos/packs para futuros empleos si tienes la información."

# O descomenta las líneas a continuación para obtener un folleto más humorístico: esto demuestra lo fácil que es incorporar el "tono":

# system_prompt = "Eres un asistente que analiza el contenido de varias páginas relevantes del sitio web de una empresa \
# y crea un folleto breve, divertido y gracioso sobre la empresa para posibles clientes, inversores y nuevos empleados. Responde en formato Markdown.\
#Incluye detalles sobre la cultura de la empresa, los clientes y los cursos/packs para futuros empleos si tienes la información."


In [ ]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"Estás mirando una empresa llamada: {company_name}\n"
    user_prompt += f"Aquí se encuentra el contenido de su página de inicio y otras páginas relevantes; usa esta información para crear un breve folleto de la empresa en Markdown.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:20_000] # Truncar si tiene más de 20.000 caracteres
    return user_prompt

In [ ]:
get_brochure_user_prompt("Anthropic", "https://anthropic.com")

Links encontrados: {'links': [{'type': 'Página Sobre nosotros', 'url': 'https://www.anthropic.com/company'}, {'type': 'Página de Carreras', 'url': 'https://www.anthropic.com/careers'}, {'type': 'Página de Investigación', 'url': 'https://www.anthropic.com/research'}, {'type': 'Página de Cursos', 'url': 'https://www.anthropic.com/learn'}, {'type': 'Página de Eventos', 'url': 'https://www.anthropic.com/events'}]}


'Estás mirando una empresa llamada: Anthropic\nAquí se encuentra el contenido de su página de inicio y otras páginas relevantes; usa esta información para crear un breve folleto de la empresa en Markdown.\nLanding page:\nTítulo de la Web:\nHome \\ Anthropic\nContenido de la Web:\nSkip to main content\nSkip to footer\nResearch\nEconomic Futures\nCommitments\nInitiatives\nTransparency\nResponsible Scaling Policy\nTrust center\nSecurity and compliance\nLearn\nLearn\nAnthropic Academy\nEngineering at Anthropic\nDeveloper docs\nCompany\nAbout\nCareers\nEvents\nNews\nTry Claude\nTry Claude\nTry Claude\nLearn more about Claude\nProducts\nClaude\nClaude Code\nClaude Developer Platform\nPricing\nContact sales\nModels\nOpus\nSonnet\nHaiku\nLog in\nClaude.ai\nClaude Console\nEN\nThis is some text inside of a div block.\nLog in to Claude\nLog in to Claude\nLog in to Claude\nDownload app\nDownload app\nDownload app\nResearch\nEconomic Futures\nCommitments\nInitiatives\nTransparency\nResponsible Sca

In [ ]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [ ]:
create_brochure("Anthropic", "https://anthropic.com")

Links encontrados: {'links': [{'type': 'Pagina Sobre nosotros', 'url': 'https://www.anthropic.com/company'}, {'type': 'Pagina de Carreras', 'url': 'https://www.anthropic.com/careers'}, {'type': 'Pagina de Cursos', 'url': 'https://www.anthropic.com/learn'}, {'type': 'Pagina de Investigación', 'url': 'https://www.anthropic.com/research'}, {'type': 'Pagina de Futuros Económicos', 'url': 'https://www.anthropic.com/economic-futures'}, {'type': 'Pagina de Transparencia', 'url': 'https://www.anthropic.com/transparency'}, {'type': 'Pagina de Eventos', 'url': 'https://www.anthropic.com/events'}]}


# Folleto de Anthropic

## Acerca de Anthropic
Anthropic es una corporación pública benéfica dedicada a la investigación y desarrollo de inteligencia artificial segura y responsable. Nuestra misión es hacer que los sistemas de IA sean confiables, interpretables y dirigibles, garantizando su beneficio a largo plazo para la humanidad. Ubicados en San Francisco, contamos con un equipo diverso de investigadores, ingenieros y expertos en políticas, entre otros.

## Cultura de la Empresa
En Anthropic, creemos que nuestra cultura es fundamental para el éxito colectivo. Nuestros valores guían cómo trabajamos juntos y cómo tomamos decisiones que impactan el futuro de la inteligencia artificial. Valoramos:

- **El bien global**: Tomamos decisiones que maximizan resultados positivos para la humanidad.
- **La transparencia**: Fomentamos un entorno de confianza y comunicación directa.
- **La competencia saludable**: Promovemos una "carrera hacia la cima" en el desarrollo seguro y confiable de IA.

## Clientes y Productos
Nuestra base de clientes incluye negocios, organizaciones sin fines de lucro y gobiernos que buscan integrar soluciones de inteligencia artificial efectivas y seguras en sus operaciones. 

**Productos destacados:**
- **Claude**: Nuestra línea de modelos de lenguaje que ayuda en diversas tareas, incluyendo programación y asistencia de agentes.
- **Claude Code**: Herramientas para mejorar la eficiencia en el desarrollo de software.
- **Claude Developer Platform**: Una plataforma para desarrolladores que permite la creación de aplicaciones y herramientas basadas en IA.

## Oportunidades de Carreras
Estamos en constante búsqueda de talentos que quieran contribuir al futuro de la IA segura. La experiencia necesaria varía según el rol, y celebramos diversas trayectorias profesionales. 

### ¿Qué ofrecemos?
- **Salud y bienestar**: Seguro médico integral, beneficios de fertilidad, y un ambiente de trabajo positivo.
- **Compensación**: Salarios competitivos y oportunidades de equidad.
- **Apoyo adicional**: Estipendios educativos y apoyo para reubicación.

### Proceso de Selección
Nuestro proceso de entrevistas está diseñado para identificar candidatos que aporten fortalezas únicas a nuestro equipo multidisciplinario. No es necesario tener experiencia previa en sistemas de aprendizaje automático para postularse; buscamos habilidades y perspectivas diversas.

## Desarrollo Futuro
Anthropic está comprometido con la investigación y el desarrollo continuo en el campo de la IA. A través de **Anthropic Academy**, ofrecemos cursos y recursos para capacitar a futuras generaciones de desarrolladores de IA y otros profesionales que deseen unirse a nosotros en esta misión.

## Contáctanos
Si quieres formar parte de un equipo que está redefiniendo el futuro de la IA, consulta nuestras posiciones abiertas en el sitio de **carreras** en nuestra página web.

---

Para más información, visita [Anthropic](https://www.anthropic.com)

In [ ]:
create_brochure("Frogames Formación", "https://cursos.frogamesformacion.com")

## Por último, una pequeña mejora

Con un pequeño ajuste, podemos cambiar esto para que los resultados se transmitan desde OpenAI,
con la animación de máquina de escribir habitual


In [ ]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("Anthropic", "https://anthropic.com")

In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")

Links encontrados: {'links': [{'type': 'Pagina Acerca de', 'url': 'https://huggingface.co/huggingface'}, {'type': 'Pagina de Carreras', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'Pagina de Precios', 'url': 'https://huggingface.co/pricing'}, {'type': 'Pagina de Documentación', 'url': 'https://huggingface.co/docs'}, {'type': 'Pagina de Blog', 'url': 'https://huggingface.co/blog'}, {'type': 'Pagina de Modelos', 'url': 'https://huggingface.co/models'}, {'type': 'Pagina de Datasets', 'url': 'https://huggingface.co/datasets'}, {'type': 'Pagina de Espacios', 'url': 'https://huggingface.co/spaces'}]}


# Folleto Breve de Hugging Face

---

## **Hugging Face – La Comunidad de IA que Construye el Futuro**

Hugging Face es la plataforma líder donde la comunidad de aprendizaje automático (ML) colabora en la creación, descubrimiento y desarrollo de modelos, conjuntos de datos y aplicaciones. Nuestra misión es democratizar el acceso a tecnología de calidad en machine learning, fomentando la colaboración entre investigadores, desarrolladores y empresas.

---

### **Cultura de la Empresa**

- **Colaborativa**: Fomentamos un ambiente abierto donde todos los miembros pueden contribuir a la creación de modelos y herramientas innovadoras.
- **Inclusiva**: Valoramos la diversidad y promovemos una comunidad donde todos puedan participar y aprender de otros.
- **Innovadora**: Estamos en constante búsqueda de soluciones que mejoren el campo de la inteligencia artificial, impulsando nuevos métodos y tecnologías.

---

### **Clientes y Socios**

Más de **50,000 organizaciones** confían en Hugging Face, incluyendo gigantes de la tecnología como:
- **Google**
- **Microsoft**
- **Amazon**
- **Meta**
- **Intel**

Nuestra plataforma es utilizada por empresas, universidades y desarrolladores individuales que buscan construir y optimizar sus proyectos de IA.

---

### **Oportunidades Profesionales**

Estamos creciendo y buscando individuos apasionados para unirse a nuestro equipo:

- **Diversas Oportunidades**: Disponemos de vacantes en áreas como investigación, desarrollo de productos, marketing, y más.
- **Crecimiento Profesional**: Fomentamos el aprendizaje continuo mediante la participación en proyectos interesantes y desafíos técnicos.
- **Trabajo Remoto**: Ofrecemos opciones de trabajo flexible, permitiendo que nuestros empleados equilibren su vida personal y profesional.

Interesados en unirse a nosotros pueden consultar nuestras oportunidades actuales [aquí](https://huggingface.co/jobs).

---

### **Cursos y Formación**

Hugging Face proporciona recursos educativos y cursos diseñados para ayudar a los futuros profesionales a desarrollar sus habilidades en IA. Algunas de las áreas que cubrimos incluyen:

- **Modelos de Lenguaje**
- **Optimización de Modelos**
- **Despliegue en Producción**

Además, ofrecemos un acceso gratuito a nuestra amplia colección de datasets y modelos a través de la plataforma Hugging Face Hub, donde también se pueden encontrar guías y documentación para aprender y experimentar.

---

### **Planificación y Precios**

Ofrecemos distintos niveles de suscripción para satisfacer las necesidades de individuos y empresas, incluyendo:
- **Pro Account**: A partir de $9/mes, incluye almacenamiento privado y créditos de inferencia.
- **Team Account**: Desde $20/usuario/mes, ideal para equipos que buscan colaboración.
- **Enterprise Solutions**: A partir de $50/usuario/mes, con características personalizadas y soporte dedicado.

### **Únete a la Comuniad**

Hugging Face no solo es un lugar para trabajar; es un espacio donde puedes aprender, compartir y crecer. Si te apasiona el machine learning y deseas ser parte de una comunidad vibrante, ¡te invitamos a registrarte y explorar nuestro Hub en [Hugging Face](https://huggingface.co)! 

--- 

**Contáctanos para más información y sé parte de la revolución en inteligencia artificial.**

--- 

*Este folleto ha sido creado para proporcionar una visión general de Hugging Face, su cultura, clientes, oportunidades de carrera y programas educativos.*

In [ ]:
stream_brochure("Frogames Formación", "https://cursos.frogamesformacion.com")

## Aplicaciones empresariales

En este ejercicio, ampliamos el código del día 1 para realizar múltiples llamadas a LLM y generar un documento.

En términos de técnicas, este es quizás el primer ejemplo de patrones de diseño de Agentic AI, ya que combinamos múltiples llamadas a LLM. Esto se abordará más en la semana 2 y luego volveremos a Agentic AI de manera importante en la semana 8, cuando construyamos una solución Agent completamente autónoma.

En términos de aplicaciones empresariales, generar contenido de esta manera es uno de los casos de uso más comunes. Al igual que con el resumen, esto se puede aplicar a cualquier vertical empresarial. Escriba contenido de marketing, genere un tutorial de producto a partir de una especificación, cree contenido de correo electrónico personalizado y mucho más. Explore cómo puede aplicar la generación de contenido a su negocio e intente crear un prototipo de prueba de concepto.